In [1]:
print("neha")

neha


In [2]:
import pandas as pd

In [3]:
data = pd.read_csv("C:\\Users\\nehas\\OneDrive\\Desktop\\Customer-Churn-Prediction\\notebooks\\processed_churn.csv")

In [4]:
data.shape

(7043, 53)

In [5]:
data.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,total_services,avg_charge,gender_Female,gender_Male,Partner_No,Partner_Yes,...,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Churn_No,Churn_Yes,tenure_group_0-1yr,tenure_group_1-2yr,tenure_group_2-4yr,tenure_group_4-6yr
0,0,1,29.85,29.85,0,14.925000,True,False,False,True,...,False,False,True,False,True,False,True,False,False,False
1,0,34,56.95,1889.50,3,53.985714,False,True,True,False,...,False,False,False,True,True,False,False,False,True,False
2,0,2,53.85,108.15,2,36.050000,False,True,True,False,...,False,False,False,True,False,True,True,False,False,False
3,0,45,42.30,1840.75,3,40.016304,False,True,True,False,...,True,False,False,False,True,False,False,False,True,False
4,0,2,70.70,151.65,1,50.550000,True,False,True,False,...,False,False,True,False,False,True,True,False,False,False


In [6]:
X = data.drop(['Churn_Yes' , 'Churn_No'] , axis = 1)
y = data['Churn_Yes']

X['TotalCharges'] = X['TotalCharges'].fillna(X['TotalCharges'].median())
X['avg_charge'] = X['avg_charge'].fillna(0)
X=X.fillna(0)

In [7]:
print(X.isnull().sum().sum())

0


In [8]:
from sklearn.model_selection import train_test_split
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size = 0.3 , random_state = 42 , stratify=y)

X_train.shape , X_test.shape

((4930, 51), (2113, 51))

In [9]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Logistic Regression requires standardization


In [10]:
print(X.isnull().sum()[X.isnull().sum() > 0])

Series([], dtype: int64)


In [11]:
print(X.isnull().sum().sum())

0


In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report , roc_auc_score
lr = LogisticRegression(max_iter=1000, random_state=42)

lr.fit(X_train_scaled, y_train)

y_prob_lr = lr.predict_proba(X_test_scaled)[:,1]
y_pred_lr = (y_prob_lr >= 0.35).astype(int)

print("LOGISTIC REGRESSION")
print(classification_report(y_test,y_pred_lr))
print("ROC-AUC :", roc_auc_score(y_test,y_prob_lr))

LOGISTIC REGRESSION
              precision    recall  f1-score   support

       False       0.89      0.80      0.84      1552
        True       0.57      0.71      0.63       561

    accuracy                           0.78      2113
   macro avg       0.73      0.76      0.74      2113
weighted avg       0.80      0.78      0.79      2113

ROC-AUC : 0.8471370389767905


In [14]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report,roc_auc_score

rf = RandomForestClassifier(n_estimators = 100 , random_state = 42 , class_weight = 'balanced')
rf.fit(X_train , y_train)
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:,1]

print("RANDOM FOREST")
print(classification_report(y_test,y_pred_rf))
print("ROC-AUC :", roc_auc_score(y_test,y_prob_rf))

RANDOM FOREST
              precision    recall  f1-score   support

       False       0.83      0.89      0.86      1552
        True       0.63      0.48      0.55       561

    accuracy                           0.79      2113
   macro avg       0.73      0.69      0.70      2113
weighted avg       0.77      0.79      0.78      2113

ROC-AUC : 0.8183495047503538


In [15]:
from xgboost import XGBClassifier
xgb = XGBClassifier(use_label_encoder = False , eval_metric = 'logloss')
xgb.fit(X_train , y_train)
y_pred_xgb = xgb.predict(X_test)
y_prob_xgb = xgb.predict_proba(X_test)[:,1]

print("XGBOOST")
print(classification_report(y_test , y_pred_xgb))
print("ROC-AUC : ", roc_auc_score(y_test , y_prob_xgb))

XGBOOST
              precision    recall  f1-score   support

       False       0.83      0.88      0.85      1552
        True       0.59      0.50      0.54       561

    accuracy                           0.78      2113
   macro avg       0.71      0.69      0.70      2113
weighted avg       0.77      0.78      0.77      2113

ROC-AUC :  0.8164354659389528


c:\Users\nehas\OneDrive\Desktop\Customer-Churn-Prediction\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [22:57:56] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [46]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, classification_report

lr = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced'
)

lr.fit(X_train_scaled, y_train)

y_prob = lr.predict_proba(X_test_scaled)[:,1]

# lower threshold
y_pred = (y_prob >= 0.27).astype(int)

print("Recall:", recall_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("ROC-AUC : ", roc_auc_score(y_test , y_prob_xgb))

Recall: 0.9376114081996435
              precision    recall  f1-score   support

       False       0.96      0.52      0.68      1552
        True       0.42      0.94      0.58       561

    accuracy                           0.63      2113
   macro avg       0.69      0.73      0.63      2113
weighted avg       0.81      0.63      0.65      2113

ROC-AUC :  0.8164354659389528


In [ ]:
import sys
print(sys.executable)

c:\Users\nehas\OneDrive\Desktop\Customer-Churn-Prediction\.venv\Scripts\python.exe


In [45]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

for t in [0.25 , 0.27 , 0.23]:
    y_pred_lr = (y_prob_lr >= t).astype(int)

    print("Threshold:", t)
    print("Accuracy :", round(accuracy_score(y_test, y_pred_lr),3))
    print("Precision:", round(precision_score(y_test, y_pred_lr),3))
    print("Recall   :", round(recall_score(y_test, y_pred_lr),3))
    print("F1 Score :", round(f1_score(y_test, y_pred_lr),3))
    print("-"*30)

Threshold: 0.25
Accuracy : 0.736
Precision: 0.502
Recall   : 0.802
F1 Score : 0.618
------------------------------
Threshold: 0.27
Accuracy : 0.747
Precision: 0.516
Recall   : 0.788
F1 Score : 0.623
------------------------------
Threshold: 0.23
Accuracy : 0.73
Precision: 0.495
Recall   : 0.822
F1 Score : 0.618
------------------------------


In [47]:
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(max_depth=5 , random_state = 42)
dt.fit(X_train , y_train)

y_pred_dt = dt.predict(X_test)
y_prob_dt = dt.predict_proba(X_test)[: , 1]

print("DECISION TREE")
print(classification_report(y_test , y_pred_dt))
print("ROC-AUC : ",roc_auc_score(y_test , y_prob_dt))

DECISION TREE
              precision    recall  f1-score   support

       False       0.86      0.86      0.86      1552
        True       0.62      0.60      0.61       561

    accuracy                           0.79      2113
   macro avg       0.74      0.73      0.74      2113
weighted avg       0.79      0.79      0.79      2113

ROC-AUC :  0.8289591258246505


In [48]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_trained_Scaled,y_train)

y_pred_knn = knn.predict(X_test_scaled)
y_prob_knn = knn.predict_proba(X_test_scaled)[:,1]

print("K NEAREST NEIGHBOUR")
print(classification_report(y_test, y_pred_knn))
print(roc_auc_score(y_test, y_prob_knn))

NameError: name 'X_trained_Scaled' is not defined

In [ ]:
from sklearn.svm import SVC

svm = SVC(probability=True)

svm.fit(X_trained_Scaled , y_train)

y_pred_svm = svm.predict(X_test_scaled)
y_prob_svm = svm.predict_proba(X_test_scaled)[:,1]

print("SVM")
print(classification_report(y_test, y_pred_svm))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_svm))

SVM
              precision    recall  f1-score   support

       False       0.82      0.92      0.87      1552
        True       0.66      0.45      0.54       561

    accuracy                           0.79      2113
   macro avg       0.74      0.68      0.70      2113
weighted avg       0.78      0.79      0.78      2113

ROC-AUC: 0.8040760470073691


In [49]:
# Selecting logistic regression as the final model because of highest recall and roc-auc-score of 84%

In [50]:
import pandas as pd

In [51]:
importance = pd.Series(lr.coef_[0], index = X_train.columns)
importance = importance.sort_values(ascending = False)
print("Top Features Increasing Churn:")
print(importance.head(10))

print("\nTop Features Reducing Churn:")
print(importance.tail(10))

Top Features Increasing Churn:
tenure_group_0-1yr             0.595800
tenure_group_4-6yr             0.508651
InternetService_Fiber optic    0.468760
TotalCharges                   0.423014
tenure_group_2-4yr             0.408754
Contract_Month-to-month        0.364873
tenure_group_1-2yr             0.339466
StreamingMovies_Yes            0.217005
StreamingTV_Yes                0.182842
total_services                 0.169559
dtype: float64

Top Features Reducing Churn:
DeviceProtection_No internet service   -0.138148
OnlineBackup_No internet service       -0.138148
OnlineSecurity_No internet service     -0.138148
InternetService_No                     -0.138148
StreamingMovies_No internet service    -0.138148
InternetService_DSL                    -0.370781
Contract_Two year                      -0.398714
MonthlyCharges                         -0.412157
tenure                                 -0.771545
avg_charge                             -0.951751
dtype: float64


In [57]:
import pickle
from sklearn.pipeline import Pipeline
pipeline = Pipeline([
    ('scaler', scaler),
    ('model' ,lr)])
pipeline.fit(X_train, y_train) 
pickle.dump(pipeline,open("pipeline.pkl","wb"))

In [59]:
import pickle
import os

# Create folder if not exists
os.makedirs("model", exist_ok=True)

# 1. Save trained model
pickle.dump(lr, open("model/model.pkl", "wb"))

# 2. Save scaler
pickle.dump(scaler, open("model/scaler.pkl", "wb"))

# 3. Save feature columns
pickle.dump(X.columns.tolist(), open("model/columns.pkl", "wb"))

# 4. Save final threshold
pickle.dump(0.27, open("model/threshold.pkl", "wb"))

print("All files saved successfully!")

All files saved successfully!


In [60]:
print(X.columns.tolist())

['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'total_services', 'avg_charge', 'gender_Female', 'gender_Male', 'Partner_No', 'Partner_Yes', 'Dependents_No', 'Dependents_Yes', 'PhoneService_No', 'PhoneService_Yes', 'MultipleLines_No', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_DSL', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_Month-to-month', 'Contract_One year', 'Contract_Two year', 'PaperlessBilling_No', 'PaperlessBilling_Yes', 'PaymentMethod_Bank transfer (aut